In [1]:
import sys
sys.path.insert(1, 'OneDrive/Documents/GitHub/PyParse')
import pandas as pd
import math
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Descriptors
from rdkit.Chem import Draw
from rdkit.Chem import PandasTools
from statistics import mean
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns

In [2]:
def getMSData(spectrum):
    """
    Takes the specific region of the rpt file pertaining to 
    m/z data for a specific peak in a specific well, and 
    returns a 2-D list containing all m/z peaks and their 
    normalised intensity.
    
    :param spectrum: Section of rpt file as string
    
    :return: 2-D list in following format: 
        [m/z value, normalised intensity of that value]
    """
    
    masses = []
    total = 0
    
    lineData = spectrum.split(";Mass\t% BPI")[1].split("\n")
    for line in lineData[1:]:
        if line == "}": #stop the for loop if end of MSData section is reached
            break
    
        massData = line.split("\t")
        if len(massData) == 2:
            floatData = [float(i) for i in massData] #convert all data to float
            masses.append(floatData)
            
            total = total + floatData[1]
    #Remove any masses which, as a percentage, round to 0 
    #to remove unnecessary baseline ions
    refined_masses = []
    for i in masses:
        if math.floor((i[1]/total)*100) > 0:
            refined_masses.append([i[0], i[1]])
    
    return refined_masses
   
def getUVData(spectrum, min_uv_threshold):
    """
    Takes the specific region of the rpt file pertaining to 
    UV absorbance spectrum data for a specific peak in a specific
    well, and returns a list containing all the maxima of that spectrum.
    The height of the maxima must be greater than the min_uv_threshold
    specified in options.
    
    :param spectrum: Section of rpt file as string
    
    :return: UV maxima as a list
    """

    UVmaxima = []
    
    lineData = spectrum.split(";Mass\t% BPI")[1].split("\n")
    UVx = []
    UVy = []
    for line in lineData:
        
        if line == "}":#stop for loop if end of UVData section is reached
            break
        
        UVdata = line.split("\t")
        if len(UVdata) == 2:
            UVx.append(float(UVdata[0]))
            UVy.append(abs(float(UVdata[1])))
    if UVy[0] > UVy[1] and UVy[0] > min_uv_threshold:
        UVmaxima.append(UVx[0])
    for i in range(1, len(UVy)-1):
        if UVy[i] > UVy[i-1] and UVy[i] > UVy[i+1] and UVy[i] > min_uv_threshold:
            UVmaxima.append(UVx[i])
    if UVy[-1] > UVy[-2] and UVy[-1] > min_uv_threshold:
        UVmaxima.append(UVx[-1])

        
    return UVmaxima

def getUserReadableWell(wellno, plate_col_no):
    """
    Converts the well as a number into a user-friendly string,
    e.g. well 11 becomes "B5" for a 4*6 well plate

    :param wellno: An integer representing a specific well on the plate
    
    :return: A string representing a specific well on the plate
    """
    
    rowVal = math.floor((wellno-1) / plate_col_no)
    colVal = (wellno) % plate_col_no
    if colVal == 0:
        colVal = plate_col_no
    
    label = f'{chr(ord("@")+(rowVal)+1)}{colVal}'
    return label

In [3]:
class rawData:
    def __init__(self, inputfile, row_no = 0, col_no = 0):
        #self.rawDADTable = pd.DataFrame(columns =['well', 'peakID', 'time', 'area', 'areaAbs', 'pStart', 'pEnd'])
        #self.rawUVTable = pd.DataFrame(columns =['well', 'peakID', 'time', 'UVvalue'])
        #self.rawMSTable = pd.DataFrame(columns =['well', 'peakID', 'time', 'MSvalue', 'MSintensity', "MStype"])
        #self.rawELSDTable = pd.DataFrame(columns =['well', 'peakID', 'time', 'area', 'areaAbs', 'pStart', 'pEnd'])
        self.row_no = row_no
        self.col_no = col_no
        
        with open(inputfile, errors = "ignore") as f:
            fullText = f.read()
            self.wellData = fullText.split("[SAMPLE]")[1:] #Split the file into individual wells
    
    def getWellFormat(self):
        #Get the plate dimensions from the first sample
        #Data sample: #Plate	01TL,XY,SD,1: 8,2:12,3: 90.0...
        #where number of rows is 8 and number of columns in 12
        #Overwrite the default option, but ensure that the user retains control
        #such that empty rows can be removed from the heatmap. 
        if self.row_no == 0 or self.col_no == 0:
            self.row_no = int(self.wellData[0].split("\n")[17].split(",")[3].split(":")[1])
            self.col_no = int(self.wellData[0].split("\n")[17].split(",")[4].split(":")[1])
            self.plate_cols_for_extraction = self.col_no
        else:
            self.plate_cols_for_extraction = int(self.wellData[0].split("\n")[17].split(",")[4].split(":")[1])
            
            
        #Find from rpt file how each well is specified 
        #Data sample: #Plate	01TL,XY,SD,1: 8,2:12,3: 90.0...
        self.row_col_order = self.wellData[0].split("\n")[17].split(",")[1]
        self.well_type = self.wellData[0].split("\n")[17].split(",")[2]
        
    def getWell(self, position):
        well_number = -1
        #If the well type is just a Single Digit...
        if self.well_type == "SD":
            #If the well is simply an integer between 1 and infinity
            #Single line function to trim full string to just the well number used
            well_number = int(self.wellData[position].split("Well")[1].split("\n")[0].split(":")[1].strip()) 
            #If the column number specified by the user is different to that found in the rpt file, 
            #this is the result of the user looking to trim off blank columns. Only wells described by 
            #a single digit need to be modified to take this into account. 
            if self.col_no != self.plate_cols_for_extraction:
                well_number = math.floor(well_number / self.plate_cols_for_extraction)*self.col_no + (well_number % self.plate_cols_for_extraction)

        #If the well type is a combination of letters/numbers...
        else:
            #Find if the well  
            if self.row_col_order == "XY":
                column = self.wellData[position].split("Well")[1].split("\n")[0].split(":")[1].split(",")[0].strip()
                row = self.wellData[position].split("Well")[1].split("\n")[0].split(":")[1].split(",")[1].strip()
            else: 
                column = self.wellData[position].split("Well")[1].split("\n")[0].split(":")[1].split(",")[1].strip()
                row = self.wellData[position].split("Well")[1].split("\n")[0].split(":")[1].split(",")[0].strip()

            #Convert the column to integer, either by direct
            #conversion, or by finding position in the alphabet
            try:
                col_as_int = int(column)
            except:
                col_as_int = ord(column.capitalize()) - 64

            #Convert the row to integer, either by direct
            #conversion, or by finding position in the alphabet
            try: 
                row_as_int = int(row)
            except: 
                row_as_int = ord(row.capitalize()) - 64

            #Calculate the wellno as a single integer
            well_number = (row_as_int - 1) * self.plate_cols_for_extraction + col_as_int
        return well_number
    
    def processDAD(self):
        peak_list = []
        for i in range(len(self.wellData)):
            functions = self.wellData[i].split("[FUNCTION]")
            well_number = self.getWell(i)
            for j in range(len(functions[1:])):
                function = functions[1:][j]
                lines = function.split("\n")
                
                #get peakarea for this peak
                if "Type\tDAD" in lines[4]:
                    chromatograms = function.split("[CHROMATOGRAM]")[1:]
                    for chromatogram in chromatograms:
                        c_lines = chromatogram.split("\n")
                        if "Description\tDAD:" in c_lines[3]:
                            spectra = function.split("[SPECTRUM]")[1:] #split by spectrum (i.e. each peak)
                            #chroma[wellno] = getChromatogram(chromatogram.split("[TRACE]")[1]) #get chromatogram for this well
                            peaks = chromatogram.split("[PEAK]")[1:]
                            for peak in peaks:
                                new_entry = {
                                    "well": well_number,
                                    "peakID": int(peak.split("Peak ID")[1].split("\n")[0].strip()),
                                    "time": float(peak.split("Time")[1].split("\n")[0].strip()),
                                    "pStart": peak.split("Peak\t")[1].split("\n")[0].split("\t")[0],
                                    "pEnd": peak.split("Peak\t")[1].split("\n")[0].split("\t")[1],
                                    "area": float(peak.split("Area %Total")[1].split("\n")[0].strip()),
                                    "areaAbs": float(peak.split("AreaAbs")[1].split("\n")[0].strip())
                                }
                                peak_list.append(new_entry)

        self.rawDADTable = pd.DataFrame(peak_list)
        
        
    def processUV(self, min_uv_threshold = 20): 
        peak_list = []
        for i in range(len(self.wellData)):
            functions = self.wellData[i].split("[FUNCTION]")
            well_number = self.getWell(i)
            for j in range(len(functions[1:])):
                function = functions[1:][j]
                lines = function.split("\n")
                if "Type\tDAD" in lines[4]:
                    spectra = function.split("[SPECTRUM]")[1:] #split by spectrum (i.e. each peak)

                    for spectrum in spectra:
                        UVdata = getUVData(spectrum, min_uv_threshold)
                        for maxima in UVdata:
                            new_entry = {
                                "well": well_number,
                                "peakID": int(spectrum.split("Peak ID")[1].split("\n")[0].strip()),
                                "time": float(spectrum.split("Time")[1].split("\n")[0].strip()),
                                "UVvalue": maxima,
                            }
                            peak_list.append(new_entry)

        if len(peak_list) == 0:
            self.rawUVTable = pd.DataFrame(columns =['well', 'peakID', 'time', 'UVvalue'])
        else:
            self.rawUVTable = pd.DataFrame(peak_list)
        
        
    def processMS(self):
        peak_list = []
        for i in range(len(self.wellData)):
            functions = self.wellData[i].split("[FUNCTION]")
            well_number = self.getWell(i)
            for j in range(len(functions[1:])):
                function = functions[1:][j]
                lines = function.split("\n")
                if "IonMode\tES" in lines[3]:
                    
                    spectra = function.split("[SPECTRUM]")[1:] #split by spectrum (i.e. each peak)
                    for spectrum in spectra:
                        if "IonMode\tES+" in lines[3]:
                            MStype = "+"
                        elif "IonMode\tES-" in lines[3]:
                            MStype = "-"

                        MSdata = getMSData(spectrum)
                        for ion in MSdata:
                            new_entry = {
                                "well": well_number,
                                "peakID": int(spectrum.split("Peak ID")[1].split("\n")[0].strip()),
                                "time": float(spectrum.split("Time")[1].split("\n")[0].strip()),
                                "MSvalue": ion[0],
                                "MSintensity": ion[1],
                                "MStype": MStype
                            }
                            peak_list.append(new_entry)

        
        self.rawMSTable = pd.DataFrame(peak_list)
        
        #Calculate the sum of the MSintensities in each peak, then calculate the of each MSintensity to this total
        total_intensities = self.rawMSTable.groupby(["well", "peakID", "MStype"]).agg(total_intensity=('MSintensity', 'sum'))
        self.rawMSTable = self.rawMSTable.join(total_intensities, on=["well", "peakID", "MStype"], rsuffix = "right")
        self.rawMSTable["perc_intensity"] = 100 * self.rawMSTable["MSintensity"] / self.rawMSTable["total_intensity"]
        

            

In [4]:
inputfile = "example_dataset/Waters/Example1/example_rpt.rpt"
#inputfile = "example_dataset/Waters/Example2/LC-MS Data for 48-Well Plate.rpt"
#inputfile = "C:/Users/joe.mason/OneDrive - Domainex/Desktop/test.rpt"
test = rawData(inputfile)

In [5]:
test.getWellFormat()

In [6]:
test.processDAD()
test.processMS()
test.processUV()

In [7]:
test.rawUVTable

,well,peakID,time,UVvalue
0,1,1,1.2300,210.95
1,1,1,1.2300,234.95
2,2,1,0.3533,209.95
3,2,2,1.2958,229.95
4,3,1,0.5863,212.95
...,...,...,...,...
70,23,1,0.4296,209.95
71,23,1,0.4296,269.95
72,24,1,1.0150,258.95
73,24,2,1.4321,209.95


In [8]:
test.rawDADTable

,well,peakID,time,pStart,pEnd,area,areaAbs
0,1,1,1.2300,1.2154,1.2525,100.00,1.071910e+06
1,2,1,0.3533,0.3400,0.3700,1.07,1.085240e+04
2,2,2,1.2958,1.2817,1.3192,98.93,1.001665e+06
3,3,1,0.5863,0.5758,0.6071,100.00,1.190068e+06
4,4,1,0.5483,0.5383,0.5692,98.21,1.185495e+06
5,4,2,1.1246,1.1109,1.1467,1.79,2.156398e+04
6,5,1,1.2246,1.2104,1.2484,97.78,6.553948e+05
7,5,2,1.3688,1.3546,1.3929,2.22,1.486370e+04
8,6,1,1.1142,1.1000,1.1379,100.00,1.121448e+06
9,7,1,1.2284,1.2138,1.2542,100.00,9.312498e+05


In [84]:
class Assignment:
    def __init__(self, filename, plate_col_no):   
        #read csv file into dataframe
        #replace empty cells with an empty string
        #convert all column names to lower case and remove whitespace
        self.inputCSV = pd.read_csv(filename)
        self.inputCSV.fillna("", inplace=True)
        self.inputCSV.columns = self.inputCSV.columns.str.strip().str.lower()
        
        self.plate_col_no = plate_col_no
        
        
    
    #Fn to convert a well name like B5 to machine format (11)
    def convertWellToNum(self, wells):
        result = []
        
        for well in wells:
            row = well[0]
            column = well[1:]
            #if the format of the row/column conforms to expectations
            if isinstance(int(column[0]), int):
                result.append(int((ord(row) - 65) * self.plate_col_no + int(column)))
            #Plates with more than 26 rows are unsupported at present. 
            else:
                logging.info("The well specified implies an unsupported plate.")
                sys.exit(2)
        return result
    
    #Fn to convert smiles into canonicalised smiles
    def getCanonSmiles(self, smiles):
        mol = Chem.MolFromSmiles(smiles.strip())
        return Chem.MolToSmiles(mol)
            
    def generateCPTable(self):
        compound_list = []
        type_dic = {
            "desired product smiles": "Product",
            "limiting reactant smiles":"Limiting Reactant",
            "internalstd smiles": "InternalSTD"
        }
        counter = {
            "desired product smiles": 1,
            "limiting reactant smiles": 1,
            "byproduct": 1
        }
        
        #Canonicalise all the incoming smiles in key columns in case the user hadn't done so already
        for col in self.inputCSV:
            if col in type_dic or ("byproduct" in col and "smiles" in col):
                self.inputCSV[col] = self.inputCSV[col].apply(self.getCanonSmiles)
        
        for col in self.inputCSV:
            if col in type_dic or ("byproduct" in col and "smiles" in col):
                cpname_column = f'{col.split(" smiles")[0]} name'
                cprt_column = f'{col.split(" smiles")[0]} rt'
                
                #group the input CSV by canonical smiles in that column, aggregated the wells into a list
                groupeddf = self.inputCSV.groupby(col, as_index=False)[["well"]].agg(lambda x: list(x))
                
                #Iterate through each of those grouped entries
                for index, row in groupeddf.iterrows():
                    cpindex = row[col]
                    cptype = type_dic[col] if col in type_dic else "byproduct"
                    #get a name for the compound if one was provided
                    name = ""
                    if cpname_column in self.inputCSV.columns:
                        name_series = self.inputCSV.loc[self.inputCSV[col] == row[col]][cpname_column]
                        potential_names = [x for x in name_series if x != ""]
                        if len(potential_names) != 0:
                            name = potential_names[0]
                    #if a name could not be generated, create a generic one using a simple counter
                    if name == "":
                        if col == "internalstd smiles":
                            name = "InternalSTD"
                        elif col in counter:
                            name = f'{type_dic[col]}{counter[col]}'
                            counter[col] = counter[col] + 1
                        elif "byproduct" in col:
                            name = f'Byproduct{counter["byproduct"]}'
                            counter["byproduct"] = counter["byproduct"] + 1
                    
                    #get a rentention time for the compound if one was provided
                    rt = 0
                    if cprt_column in self.inputCSV.columns:
                        rt_series = self.inputCSV.loc[self.inputCSV[col] == row[col]][cprt_column]
                        potential_rt = [x for x in rt_series if x != ""]
                        if len(potential_rt) != 0:
                            rt = potential_rt[0]
                            
                    new_entry = {
                        "smiles": cpindex,
                        "type": cptype,
                        "locations": row["well"],
                        "name": name,
                        "rt": rt,
                        "comments": []
                    }
                    compound_list.append(new_entry)
        
        self.cpTable = pd.DataFrame(compound_list)
        
        #set the index to be the canonicalised smiles
        self.cpTable.index = list(self.cpTable["smiles"])
        
        #convert the "A1" style well IDs into a integer, to allow matching to a well in the rawData
        self.cpTable["locations"] = self.cpTable["locations"].apply(self.convertWellToNum)
    

    
    def generateEMs(self, calc_boc):
        
        def getMW(smiles):
            mol = Chem.MolFromSmiles(smiles)
            return round(Descriptors.ExactMolWt(mol), 2)
        
        def transform_and_getMW(smiles, smirks, stage):
            try:
                mol = Chem.MolFromSmiles(smiles)
                rxn1 = AllChem.ReactionFromSmarts(smirks)
                new_mol1 = rxn1.RunReactants((mol, ))[0][0]
                #Sanitise the molecule to make sure that a sensible molecule was produced. 
                Chem.SanitizeMol(new_mol1)
                return round(Descriptors.ExactMolWt(new_mol1), 2)
            except:
                if ("Cl" in smiles or "Br" in smiles) and stage == "mass2":
                    mol = Chem.MolFromSmiles(smiles)
                    return round(Descriptors.ExactMolWt(mol), 2) + 2
                else:
                    return 0
            
        self.cpTable["mass1"] = self.cpTable["smiles"].apply(lambda smiles: getMW(smiles))
        
        if calc_boc == "True":
            
            smirks1 = "[NX3,n:1][C:2](=[O:3])[O:4][C]([CH3])([CH3])[CH3]>>[*:1][C:2](=[O:3])[O:4]"
            self.cpTable["mass2"] = self.cpTable["smiles"].apply(lambda smiles: transform_and_getMW(smiles, smirks1, "mass2"))
            
            smirks2 = "[NX3,n:1][C](=[O])[O][C]([CH3])([CH3])[CH3]>>[*:1][H]"
            self.cpTable["mass3"] = self.cpTable["smiles"].apply(lambda smiles: transform_and_getMW(smiles, smirks2, "mass3"))
            

    def findHits(self, msData, mass_abs_tol = 0.5, min_massconf_threshold = 10, 
                                      time_abs_tol = 0.025, calc_higherions = "True"):
        
        def getMatches(compound):
            df = msData.loc[msData["well"].isin(compound["locations"])]
            
            MS_hits = []
            
            for mass in [compound["mass1"], compound["mass2"], compound["mass3"]]:
                hits = df.loc[(df["MStype"] == "+") & 
                                   ((abs(df["MSvalue"] - (mass + 1.01)) <= mass_abs_tol) |
                                    (abs(df["MSvalue"] - (mass + 2.02)/2) <= mass_abs_tol) |
                                    (abs(df["MSvalue"] - (mass + 3.03)/3) <= mass_abs_tol))]
                MS_hits = MS_hits + list(hits.index.values)
                
                hits = df.loc[(df["MStype"] == "-") & 
                                   (abs(df["MSvalue"] - (mass - 1.01)) <= mass_abs_tol)]
                MS_hits = MS_hits + list(hits.index.values)
            
            grouped = df[df.index.isin(MS_hits)].groupby(["well", "peakID"], as_index = False).agg(mass_conf = ("perc_intensity", "sum"))
            
            grouped = grouped.loc[grouped["mass_conf"] >= min_massconf_threshold]

            return grouped.to_dict("records")                      

        self.cpTable["hits"] = self.cpTable.apply(getMatches, axis = 1)
    
    def validateHits(self, lcData, msData, uvData, time_abs_tol = 0.025, massconf_threshold = 0.5, uv_abs_tol = 10,
                    uv_cluster_threshold = 0.5, uv_match_threshold = 0.5, cluster_size_threshold = 0.8, min_no_of_wells = 5,
                    validate = "True"):
        
        def getRelevantPeaks(x, data):
            
            return list(data[(data["well"] == x["well"]) & (data["peakID"] == x["peakID"])].index.values)
            
        def clusterHits(compound):
                
            relevant_indexes = []
            for hit in compound["hits"]:
                relevant_indexes = relevant_indexes + getRelevantPeaks(hit, lcData)
            
            df = lcData[lcData.index.isin(relevant_indexes)]
            df.sort_values("time", inplace = True)
            
            clusters = []
            for index in df.index:
                if len(clusters) == 0:
                    clusters.append([index])
                else:
                    clusterFound = False
                    for cluster in clusters:
                        mean_rt = mean([df.loc[i, "time"] for i in cluster])

                        if abs(mean_rt - df.loc[index, "time"]) < time_abs_tol:
                            cluster.append(index)
                            clusterFound = True
                            break
                    if not clusterFound:
                        clusters.append([index])
            
            #At this point, clusters contains the indexes of the relevant rows of lcData
            return clusters
        
        def getClusterBand(compound):
            clusterbands = []
            for cluster in compound["clusters"]:
                mean = lcData.loc[lcData.index.isin(cluster)]["time"].mean()
                clusterbands.append(round(mean, 5))
            return clusterbands
                       
        def selectCluster_ifrt(row):
            comments = row["comments"]
            #If the user has specified a retention time, we should select only the cluster 
            #that is closest to that retention time, and within the specified time_abs_tol
            if row["rt"] != 0:
                suitable_clusters = [index for index, i in enumerate(row["cluster_bands"]) if abs(i - row["rt"]) < time_abs_tol]

                #If there is more than one cluster close to the specified retention time
                #take only the cluster which is closest 
                if len(suitable_clusters) > 1:
                    diffs = [row["cluster_bands"][index]-row["rt"] for index in suitable_clusters]
                    index_min = min(range(len(diffs)), key=diffs.__getitem__)
                    row["clusters"] = [row["clusters"][suitable_clusters[index_min]]]
                    #Update the cluster bands to only include the correct label
                    cluster_bands = [row["cluster_bands"][suitable_clusters[index_min]]]

                    row["comments"].append("<strong>Multiple clusters were found close the specified"
                                        " retention time.</strong>")
                    row["comments"].append(f'<strong>Cluster {index_min} was selected as it was closest'
                                        ' to the specified retention time.</strong>')
                elif len(suitable_clusters) == 1:
                    row["clusters"] = [row["clusters"][suitable_clusters[0]]]
                    row["cluster_bands"] = [row["cluster_bands"][suitable_clusters[0]]]
                    row["comments"].append("<strong>A single cluster was found close the specified"
                                        " retention time and this was selected for analysis.</strong>")
                else:
                    row["comments"].append("<strong>No cluster was found near to the specified "
                                        "retention time. Proceeding with analysis using all "
                                        f'{len(row["clusters"])} clusters.</strong>')
            return row
            
        def refineClustersByTime(row):
            """
            Takes in input cluster of all the hit peaks, 
            and refines them by finding a mid-value for the retention
            time based on which hit has the greatest number of nearest neighbours. 
            Sorts the best hits into "green", uncertain ones into "orange" 
            and those where another peak closer to the mid-value was found
            in the same well into "discarded". 

            :param cluster: list of dictionaries, where each dictionary is a hit
            :param comments: A list of comments for that structure so far.

            :return: List comprising [a dictionary for the refined cluster, list of comments]
            """

            [clusters, comments, expected_rt] = [row["clusters"], row["comments"], row["rt"]]
            refined_clusters = []

            for cluster in clusters:
                refined_cluster = {
                    "green":[],
                    "orange": [],
                    "discarded": [],
                }

                mid_values = []
                mid_value = 0

                if expected_rt != 0:
                    mid_value = expected_rt
                else: 
                    for i in cluster:
                        mid_value = lcData.loc[i, "time"]
                        mid_values.append([mid_value, len([lcData.loc[j, "time"] for j in cluster 
                                                           if abs(lcData.loc[j, "time"] - mid_value) < time_abs_tol/4])])
                    mid_value = max(mid_values, key = lambda x: x[1])[0]

                #sort the peaks by the well they occupy
                peaks_by_wells = {}
                for i in cluster:
                    if lcData.loc[i, "well"] not in peaks_by_wells:
                        peaks_by_wells[lcData.loc[i, "well"]] = []
                    peaks_by_wells[lcData.loc[i, "well"]].append(i)

                #For each well, select the peak that's closest to the mid-value in cases
                #where there was more than one hit in that cluster in one well
                #Use peakAdded to ensure that only a single peak per well is added to green, in the 
                #unlikely case that there are two peaks in the same well with the same retention time
                #(i.e. LCMS machine processing error)
                for index, well in peaks_by_wells.items():
                    if len(well) > 1:
                        min_diff = min([abs(lcData.loc[i, "time"]-mid_value) for i in well])
                        peakAdded = False
                        for i in well:

                            if abs(lcData.loc[i, "time"]-mid_value) == min_diff and not peakAdded:
                                refined_cluster["green"].append(i)
                                peakAdded = True
                            else:
                                refined_cluster["discarded"].append(i)
                                row["comments"].append(f'Peak at {lcData.loc[i, "time"]} '
                                            f'in well {getUserReadableWell(index, self.plate_col_no)} was discarded '
                                            'as there was an alternative peak '
                                            'in the same well which was closer to the '
                                            'mid-point of the cluster.')
                    else:
                        refined_cluster["green"].append(well[0])

                #Refine these hits further by finding those which are within time_abs_tol/2
                #of the mid-value. Any others are marked as tentative and the user is alerted.         
                ref2_cluster = {
                    "green":[],
                    "orange": refined_cluster["orange"],
                    "discarded": refined_cluster["discarded"],
                }        
                
                for i in refined_cluster["green"]:
                    if abs(lcData.loc[i, "time"] - mid_value) < time_abs_tol / 2:
                        ref2_cluster["green"].append(i)
                    else:
                        ref2_cluster["orange"].append(i)
                        comments.append(f'Peak at {lcData.loc[i, "time"]} in '
                                       f'well {getUserReadableWell(lcData.loc[i, "well"], self.plate_col_no)} was '
                                       'marked as tentative as it was found to be too '
                                       'far from the mid-value of the cluster.')
                refined_clusters.append(ref2_cluster)

            row["clusters"] = refined_clusters
            return row
        
        def refineClustersByMassConf(row):
            """
            Takes in input cluster of all the hit peaks, 
            and refines them by ensuring all peaks have a similar mass confidence
            to the cluster's mean. Those which do are left in "green"; 
            those which don't are moved to the "orange" category.

            :param cluster: a dict, with list of dicts for each header
            :param comments: A list of comments for the compound so far

            :return: List comprising [a dictionary for the refined cluster, list of comments]
            """
            refined_clusters = []
            for cluster in row["clusters"]:
                refined_cluster = {
                    "green": [],
                    "orange": cluster["orange"],
                    "discarded": cluster["discarded"],
                }
                if len(cluster["green"]) > 0:
                    #Get a dictionary of the mass_conf for each peak, indexed by the index present in lcData
                    mass_conf_dict = {}
                    for i in cluster["green"]:
                        well = lcData.loc[i, "well"]
                        peakID = lcData.loc[i, "peakID"]
                        for j in row["hits"]:
                            if j["well"] == well and j["peakID"] == peakID:
                                mass_conf_dict[i] = j["mass_conf"]

                    #find what the mean mass confidence is of all peaks
                    #currently under the "green" category
                    total = sum(mass_conf_dict[x] for x in cluster["green"])
                    mean_mass_conf = total / len(cluster["green"])

                    for i in cluster["green"]:
                        if mass_conf_dict[i] < mean_mass_conf * massconf_threshold:
                            refined_cluster["orange"].append(i)
                            row["comments"].append(f'Peak at {lcData.loc[i, "time"]} in well '
                                            f'{getUserReadableWell(lcData.loc[i, "well"], self.plate_col_no)} '
                                            'was marked tentative due to poor mass '
                                            'confidence in comparison to the rest of the cluster.')
                        else:
                            refined_cluster["green"].append(i)
                refined_clusters.append(refined_cluster)
            row["clusters"] = refined_clusters
            return row
        
        def refineClustersByUV(row):
            """
            Takes in input cluster of all the hit peaks, 
            and refines the cluster by ensuring all peaks have a similar set
            of UV maxima. Those which do are left in "green", those which
            don't are moved to the "orange" category

            :param cluster: a dict, with list of dicts for each header
            :param UVdatafound: boolean for whether the rpt data contains UV data
            :param comments: A list of comments for that structure so far

            :return: List comprising [a dictionary for the refined cluster, list of comments]
            """
            refined_clusters = []
            
            for cluster in row["clusters"]:
                refined_cluster = {
                    "green": [],
                    "orange": cluster["orange"],
                    "discarded": cluster["discarded"],
                }

                if len(cluster["green"]) > 0:
                    

                    UVclusters = []

                    for i in cluster["green"]:
                        for UV in uvData.loc[(uvData["well"] == lcData.loc[i, "well"]) & (uvData["peakID"] == lcData.loc[i, "peakID"])]["UVvalue"]:
                            if len(UVclusters) == 0:
                                UVclusters.append([UV])

                            else:
                                clusterFound = False
                                for UVcluster in UVclusters:
                                    if abs(UVcluster[-1] - UV) < uv_abs_tol:
                                        UVcluster.append(UV)
                                        clusterFound = True
                                        break
                                if not clusterFound:
                                    UVclusters.append([UV])

                    meanUV = []
                    #Check to ensure UVclusters isn't an empty set
                    if len(UVclusters) > 0:
                        lengthOfMostCommon = max([len(UVcluster) for UVcluster in UVclusters])

                        for UVcluster in UVclusters:
                            if len(UVcluster) >= lengthOfMostCommon * uv_cluster_threshold:
                                meanUV.append(mean(UVcluster))

                        for i in cluster["green"]:
                            UVvalues = (uvData.loc[(uvData["well"] == lcData.loc[i, "well"]) 
                                               & (uvData["peakID"] == lcData.loc[i, "peakID"])]["UVvalue"].values)
                            intersection = []
                            for meanvalue in meanUV:
                                for UV in UVvalues:
                                    if abs(meanvalue - UV) < uv_abs_tol:
                                        intersection.append(meanvalue)

                            if len(intersection) >= len(meanUV) * uv_match_threshold:
                                refined_cluster["green"].append(i)
                            else:
                                refined_cluster["orange"].append(i)
                                row["comments"].append(f'Peak at {lcData.loc[i, "time"]} in well '
                                                f'{getUserReadableWell(lcData.loc[i, "time"], self.plate_col_no)} '
                                                'was marked tentative due to mismatch in UV maxima with the rest of the cluster.')
                    #If UVclusters is an empty set, pass through the original hits without further
                    #validation. 
                    else:
                        row["comments"].append('UV validation not permitted where cluster contains peaks with no UV data. '
                            'UV validation was not performed.')
                        refined_cluster["green"] = cluster["green"]
                else:
                    row["comments"].append('UV data was not found for the plate, so UV validation was not performed.')
                    refined_cluster["green"] = cluster["green"]
                refined_clusters.append(refined_cluster)
            
            row["clusters"] = refined_clusters
            return row
        
        def selectClusterByMassConf(row):
            """
            If more than one cluster was found for the compound, 
            this function is called to try to select a single cluster based on 
            which cluster has the highest mean massConf. If more than one cluster
            has a close-to-highest-mean massconf, take them all. 

            :param clusters: a list of dictionaries, with list of dictionaries for each header
            :return refined_clusters: a list of dictionaries, with list of dictionaries for each header

            :return discarded_clusters: a list of dictionaries, with list of dictionaries for each header
            """
            
            refined_clusters = []
            discarded_clusters = []

            means = []
            #find the mean massconf for each cluster
            for cluster in row["clusters"]:
                if len(cluster["green"]) > 0:
                    #Get a dictionary of the mass_conf for each peak, indexed by the index present in lcData
                    mass_conf_dict = {}
                    for i in cluster["green"]:
                        well = lcData.loc[i, "well"]
                        peakID = lcData.loc[i, "peakID"]
                        for j in row["hits"]:
                            if j["well"] == well and j["peakID"] == peakID:
                                mass_conf_dict[i] = j["mass_conf"]
                    
                    #Calculate the mean mass_conf of the cluster
                    mean_mass_conf = sum([mass_conf_dict[i] for i in cluster["green"]]) / len(cluster["green"])
                    means.append(mean_mass_conf)
                else:
                    means.append(0)
                    
            #find the maximum mean value to compare all clusters against
            max_mean = max(means)
            
            #Filter the clusters by those which have a mean mass confidence that is at least the specified
            #percentage of the maximum observed mean mass confidence (set by massconf_threshold, typically 80%)
            for i in range(len(means)):
                if max_mean * massconf_threshold < means[i] or (max_mean == 0 and means[i] == 0):
                    refined_clusters.append(row["clusters"][i])
                else:
                    discarded_clusters.append(row["clusters"][i])
                    
            row["refined_clusters"] = refined_clusters
            row["discarded_clusters"] = discarded_clusters
            return row

        def selectClusterBySize(row):
            """
            If more than one cluster was found for the compound, 
            this function is called to try to select a single cluster based on 
            which cluster is the largest. If more than one cluster
            has a close-to-largest size, take them all. 

            :param clusters: a list of dictionaries, with list of dictionaries for each header
            :return refined_clusters: a list of dictionaries, with list of dictionaries for each header

            :return discarded_clusters: a list of dictionaries, with list of dictionaries for each header
            """

            refined_clusters2 = []
            discarded_clusters = row["discarded_clusters"]

            lengths = [len(cluster["green"]) + len(cluster["orange"]) for cluster in row["refined_clusters"]]
            max_length = max(lengths)

            for cluster in row["refined_clusters"]:
                if len(cluster["green"]) + len(cluster["orange"]) > max_length * cluster_size_threshold:
                    refined_clusters2.append(cluster)
                else:
                    discarded_clusters.append(cluster)

            #If there is still more than one cluster, reset the process using only those peaks
            #that haven't been marked as suspicious (orange)
            if len(refined_clusters2) > 1:

                lengths = [len(cluster["green"]) for cluster in row["refined_clusters"]]
                max_length = max(lengths)

                #As long as at least one cluster had a green hit, filter the clusters by size
                if max_length != 0:
                    refined_clusters2 = []
                    discarded_clusters = row["discarded_clusters"]

                    for cluster in row["refined_clusters"]:
                        if len(cluster["green"]) > max_length * cluster_size_threshold:
                            refined_clusters2.append(cluster)
                        else:
                            discarded_clusters.append(cluster)
            row["refined_clusters"] = refined_clusters2
            row["discarded_clusters"] = discarded_clusters
            return row
        
        def indexClusterByWells(row):
            cluster_by_well = {}
            for cluster in row["refined_clusters"]:
                for i in cluster["green"]:
                    if lcData.loc[i, "well"] not in cluster_by_well:
                        cluster_by_well[lcData.loc[i, "well"]] = {
                                "green": [],
                                "orange": [],
                                "discarded": [],
                                }

                    cluster_by_well[lcData.loc[i, "well"]]["green"].append(i)

                for i in cluster["orange"]:
                    if lcData.loc[i, "well"] not in cluster_by_well:
                        cluster_by_well[lcData.loc[i, "well"]] = {
                                "green": [],
                                "orange": [],
                                "discarded": [],
                                }

                    cluster_by_well[lcData.loc[i, "well"]]["orange"].append(i)

                for i in cluster["discarded"]:
                    if lcData.loc[i, "well"] not in cluster_by_well:
                        cluster_by_well[lcData.loc[i, "well"]] = {
                                "green": [],
                                "orange": [],
                                "discarded": [],
                                }

                    cluster_by_well[lcData.loc[i, "well"]]["discarded"].append(i)
                    
            row["clusters_indexed_by_well"] = cluster_by_well
            return row
            
        def refine_and_select(row): 
            #If there are sufficient wells to perform refine and select a cluster, do so. 
            #Otherwise, simply mark all hits as "green" fill in the necessary table structure. 
            if len(row["hits"]) > min_no_of_wells and validate == "True":
                row = refineClustersByTime(row)
                row = refineClustersByMassConf(row)
                row = refineClustersByUV(row)
                row = selectClusterByMassConf(row)
                row = selectClusterBySize(row)
            else:
                refined_clusters = []
                for cluster in row["clusters"]:
                    refined_cluster = {
                        "green": [],
                        "orange": [], 
                        "discarded": []
                    }
                    for i in cluster:
                        refined_cluster["green"].append(i)
                    refined_clusters.append(refined_cluster)
                row["refined_clusters"] = refined_clusters
                row["discarded_clusters"] = []
                
                if validate != "True":
                    row["comments"].append("Validation was not performed as requested by the user.")
                else:
                    row["comments"].append(f'Validation was not performed for {row["name"]} as '
                                        'there were insufficient hits.')
            return row
        
        def finalResult(row):
            final_result = {
                "green": [],
                "discarded": []    
            }
            
            for well in row["clusters_indexed_by_well"]:
                
                output["discarded"] += well["discarded"]
                if len(well["green"]) > 0:
                    if len(well["green"]) == 1:
                        output["green"].append(well["green"][0])
                    else:
                        id_max = options.mass_or_area

                        max_val = max(peak[id_max] for peak in well["green"])
                        peak_added = False

                        #Sort the peaks by their peak area, so that if two
                        #peaks have the same mass_conf when options.mass_or_area is
                        #equal to "mass_conf", the largest peak is selected in preference. 

                        for peak in sorted(well["green"], key = lambda x: x["area"], reverse=True):
                            if peak_added == False and peak[id_max] == max_val:
                                output["green"].append(peak)
                                comment_text.append(f'Largest {id_max} selected in preference to others available for well '
                                                    f'{getUserReadableWell(peak["well"])}.')
                                peak_added = True
                            else:
                                output["discarded"].append(peak)
                                comment_text.append(f'The peak at {peak["time"]} in well {getUserReadableWell(peak["well"])} '
                                                          f'was discarded as it had a smaller {id_max} than an '
                                                          f'equally likely alternative.')

                    if len(well["orange"]) > 0:
                        output["discarded"] += well["orange"]
                        for peak in well["orange"]:
                            comment_text.append(f'The peak at {peak["time"]} in well '
                                        f'{getUserReadableWell(peak["well"])} for {cpname} was discarded '
                                        f'because a better match was found.')
                elif len(well["orange"]) == 1:
                    peak = well["orange"][0]
                    output["green"].append(well["orange"][0])
                    comment_text.append(f'<strong>The tentative peak at {peak["time"]} in well {getUserReadableWell(peak["well"])} was '
                                              f'used as there was no better option. User should check this well.</strong>')
                elif len(well["orange"]) > 1:
                    id_max = options.mass_or_area
                    max_val = max(peak[id_max] for peak in well["orange"])
                    peak_added = False

                    #Sort the peaks by their peak area, so that if two
                    #peaks have the same mass_conf when options.mass_or_area is
                    #equal to "mass_conf", the largest peak is selected in preference. 

                    for peak in sorted(well["orange"], key = lambda x: x["area"], reverse=True):
                        if peak_added == False and peak[id_max] == max_val:
                            output["green"].append(peak)
                            comment_text.append(f'Largest {id_max} selected in preference to other tentative hits available for well '
                                                f'{getUserReadableWell(peak["well"])}.')
                            peak_added = True
                        else:
                            output["discarded"].append(peak)
                            comment_text.append(f'<strong>The tentative peak at {peak["time"]} in well {getUserReadableWell(peak["well"])} '
                                                          f'was discarded as it had a smaller {id_max} than an '
                                                          f'equally likely alternative.</strong>')
        

                
            
        self.cpTable["clusters"] = self.cpTable.apply(clusterHits, axis = 1)
        self.cpTable["cluster_bands"] = self.cpTable.apply(getClusterBand, axis = 1)
        self.cpTable = self.cpTable.apply(selectCluster_ifrt, axis = 1)
        
        self.cpTable = self.cpTable.apply(refine_and_select, axis = 1)
        
        self.cpTable = self.cpTable.apply(indexClusterByWells, axis = 1)
        
        
        
        
        
        
            
        
        
            
            
                
            
            
        
        
    

In [85]:
#cpTable = Assignment("example_dataset/Waters/Example2/PyParse_designer_platemap.csv", 12)
cpTable = Assignment("example_dataset/Waters/Example1/example_platemap.csv", 12)

In [86]:
cpTable.generateCPTable()

In [87]:
cpTable.generateEMs("True")

In [88]:
msData = test.rawMSTable
uvData = test.rawUVTable
detector = "UV"

if detector == "UV":  
    lcData = test.rawDADTable
else:
    lcData = test.rawELSDTable

In [89]:
cpTable.findHits(msData)

In [90]:
cpTable.validateHits(lcData, msData, uvData)

In [91]:
cpTable.cpTable

,smiles,type,locations,name,rt,comments,mass1,mass2,mass3,hits,clusters,cluster_bands,refined_clusters,discarded_clusters,clusters_indexed_by_well
Brc1ccc(Br)c2ncccc12,Brc1ccc(Br)c2ncccc12,Product,[20],Product1,0,[Validation was not performed for Product1 as ...,284.88,286.88,0.00,"[{'well': 20, 'peakID': 1, 'mass_conf': 68.539...",[[28]],[1.1763],"[{'green': [28], 'orange': [], 'discarded': []}]",[],"{20: {'green': [28], 'orange': [], 'discarded'..."
Brc1ccc2cccc(Br)c2n1,Brc1ccc2cccc(Br)c2n1,Product,[5],Product2,0,[Validation was not performed for Product2 as ...,284.88,286.88,0.00,"[{'well': 5, 'peakID': 1, 'mass_conf': 73.7592...",[[6]],[1.2246],"[{'green': [6], 'orange': [], 'discarded': []}]",[],"{5: {'green': [6], 'orange': [], 'discarded': ..."
Brc1cccnc1N1CCOCC1,Brc1cccnc1N1CCOCC1,Product,[21],Product3,0,[Validation was not performed for Product3 as ...,242.01,244.01,0.00,"[{'well': 21, 'peakID': 1, 'mass_conf': 94.188...",[[30]],[0.9542],"[{'green': [30], 'orange': [], 'discarded': []}]",[],"{21: {'green': [30], 'orange': [], 'discarded'..."
Brc1cnc2ccccc2c1,Brc1cnc2ccccc2c1,Product,[16],Product4,0,[Validation was not performed for Product4 as ...,206.97,208.97,0.00,"[{'well': 16, 'peakID': 1, 'mass_conf': 77.832...",[[22]],[1.0654],"[{'green': [22], 'orange': [], 'discarded': []}]",[],"{16: {'green': [22], 'orange': [], 'discarded'..."
CC(C)(C)OC(=O)N1CCN(C(=O)OCC2c3ccccc3-c3ccccc32)CC1C(=O)O,CC(C)(C)OC(=O)N1CCN(C(=O)OCC2c3ccccc3-c3ccccc3...,Product,[17],Product5,0,[Validation was not performed for Product5 as ...,452.19,396.13,352.14,"[{'well': 17, 'peakID': 1, 'mass_conf': 38.391...",[[23]],[0.9017],"[{'green': [23], 'orange': [], 'discarded': []}]",[],"{17: {'green': [23], 'orange': [], 'discarded'..."
CC(C)(C)OC(=O)N1CCN(c2ccc(Br)cn2)CC1,CC(C)(C)OC(=O)N1CCN(c2ccc(Br)cn2)CC1,Product,[13],Product6,0,[Validation was not performed for Product6 as ...,341.07,285.01,241.02,"[{'well': 13, 'peakID': 1, 'mass_conf': 44.446...","[[17], [18]]","[1.3217, 1.4329]","[{'green': [17], 'orange': [], 'discarded': []...",[],"{13: {'green': [17, 18], 'orange': [], 'discar..."
CC(C)(C)OC(=O)N1CCN(c2ccnc(Cl)n2)CC1,CC(C)(C)OC(=O)N1CCN(c2ccnc(Cl)n2)CC1,Product,[18],Product7,0,[Validation was not performed for Product7 as ...,298.12,242.06,198.07,"[{'well': 18, 'peakID': 1, 'mass_conf': 64.858...","[[25], [26]]","[1.0684, 1.2758]","[{'green': [25], 'orange': [], 'discarded': []...",[],"{18: {'green': [25, 26], 'orange': [], 'discar..."
CC(C)(C)OC(=O)N1CCN(c2ccnc3[nH]ccc23)CC1,CC(C)(C)OC(=O)N1CCN(c2ccnc3[nH]ccc23)CC1,Product,[14],Product8,0,[Validation was not performed for Product8 as ...,302.17,246.11,202.12,"[{'well': 14, 'peakID': 2, 'mass_conf': 86.461...",[[20]],[1.0396],"[{'green': [20], 'orange': [], 'discarded': []}]",[],"{14: {'green': [20], 'orange': [], 'discarded'..."
CC(C)(C)OC(=O)N1CCN(c2ncc(Br)cn2)CC1,CC(C)(C)OC(=O)N1CCN(c2ncc(Br)cn2)CC1,Product,[15],Product9,0,[Validation was not performed for Product9 as ...,342.07,286.01,242.02,"[{'well': 15, 'peakID': 1, 'mass_conf': 45.498...",[[21]],[1.3221],"[{'green': [21], 'orange': [], 'discarded': []}]",[],"{15: {'green': [21], 'orange': [], 'discarded'..."
CCc1ccc2cc(C(=O)O)cnc2c1,CCc1ccc2cc(C(=O)O)cnc2c1,Product,[3],Product10,0,[Validation was not performed for Product10 as...,201.08,0.00,0.00,"[{'well': 3, 'peakID': 1, 'mass_conf': 156.664...",[[3]],[0.5863],"[{'green': [3], 'orange': [], 'discarded': []}]",[],"{3: {'green': [3], 'orange': [], 'discarded': ..."


In [54]:
grouped = (msData.loc[(msData["well"] == 1) & (msData["peakID"] == 1)]
           .groupby(["well", "peakID"], as_index = False)
          .agg(mass_conf = ("perc_intensity", "sum")))

In [143]:
grouped

,well,peakID,mass_conf
0,1,1,200.0


In [144]:
msData.loc[(msData["well"] == 1) & (msData["peakID"] == 1)]

,well,peakID,time,MSvalue,MSintensity,MStype,total_intensity,perc_intensity
0,1,1,0.5019,114.09,8.840,+,218.289,4.049677
1,1,1,0.5019,121.13,16.400,+,218.289,7.512976
2,1,1,0.5019,195.16,100.000,+,218.289,45.810829
3,1,1,0.5019,196.28,11.140,+,218.289,5.103326
4,1,1,0.5019,212.20,4.815,+,218.289,2.205791
5,1,1,0.5019,231.24,10.461,+,218.289,4.792271
6,1,1,0.5019,232.25,4.217,+,218.289,1.931843
7,1,1,0.5019,239.14,3.170,+,218.289,1.452203
8,1,1,0.5019,241.22,45.503,+,218.289,20.845301
9,1,1,0.5019,242.19,6.064,+,218.289,2.777969
